## Predicting suitability for new locations

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
from IPython.display import display, HTML
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix
from decimal import Decimal
import geopandas as gpd
from shapely.geometry import Point
from scipy.interpolate import griddata
import rasterio
from rasterio.transform import from_origin

# Get multiple outputs in the same cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Ignore all warnings
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings(action='ignore', category=DeprecationWarning)
pd.set_option('display.max_columns', None)

In [ ]:
# I can use the points of absence to predict new location??
# Load new data for prediction (adjust as needed)
# new_locations = pd.read_csv('C:/Users/charl/Desktop/__Thesis - Research/Github/vineyard-establishment-using-xgboost/points_data/points_of_absence_test_new.csv', delimiter=';')
# new_locations_scaled = scaler.transform(new_locations)

# Make predictions
# suitability_scores = xgb_model.predict(new_locations)

# Add predictions to the new_locations dataframe
# new_locations['suitability_score'] = suitability_scores

# Save results
# new_locations.to_csv('predicted_suitability.csv', index=False)

In [ ]:
new_locations_df = pd.read_csv('C:/Users/charl/Desktop/__Thesis - Research/Github/vineyard-establishment-using-xgboost/input_points_data/temp_mean_raster_to_points_wc_2.csv', delimiter=';')

print("New locations", new_locations_df.shape)

In [ ]:
print("New locations")
new_locations_df.head()
new_locations_df.describe()

In [ ]:
new_locations_df.isnull().sum()

In [ ]:
new_locations_df.dtypes

In [ ]:
# Remove commas from the values and convert to float/double
def clean_and_convert(x):
    return x.str.replace(',', '.').astype('float')

# Converting the columns to float
columns_to_convert_for_new_locations = ['POINT_X','POINT_Y','sudem_1000m_albers_filled_wc', 'sudem_1000m_albers_potential_solar_rad_wh2m_wc',
                        'sudem_1000m_albers_twi_d8_wc', 'sudem_1000m_albers_slope_degree_wc', 'sudem_1000m_albers_slope_wc',
                        'sudem_1000m_albers_aspect_wc', 'sudem_1000m_albers_flow_network_wc', 'sudem_1000m_albers_curvature_wc',
                        'composite_temp_mean_1000m_k30_wc','composite_temp_min_1000m_k30_wc', 'composite_temp_max_1000m_k30_wc',
                        'composite_rainfall_sum_1000m_k30_wc', 'composite_gdd_grapes_1000m_wc', 'composite_AETI_1000m_albers_wc',
                        'nitrogen_0_to_5cm_1000m_albers_wc', 'sand_0_to_5cm_1000m_albers_wc', 'soc_0_to_5cm_1000m_albers_wc',
                        'clay_content_0_to_5cm_1000m_albers_wc', 'soil_ph_0_to_5cm_1000m_albers_wc', 'silt_0_to_5cm_1000m_albers_wc',
                        'ndvi_wc', 'euclidean_distance_roads_1000m_albers_wc', 'euclidean_distance_rivers_1000m_albers_wc']

for col in columns_to_convert_for_new_locations:
    new_locations_df[col] = clean_and_convert(new_locations_df[col])

In [ ]:
# checking the data types again

new_locations_df.dtypes

In [ ]:
new_locations_df

# creating a new df called new_locations_df_2 to use for the csv file
new_locations_df_2 = new_locations_df.copy()

new_locations_df_2

In [ ]:
# interpolate the columns with null values

columns_to_interpolate_for_new_locations = ['sudem_1000m_albers_filled_wc', 'sudem_1000m_albers_potential_solar_rad_wh2m_wc',
                                            'sudem_1000m_albers_twi_d8_wc', 'sudem_1000m_albers_slope_degree_wc', 'sudem_1000m_albers_slope_wc',
                                            'sudem_1000m_albers_aspect_wc', 'sudem_1000m_albers_flow_network_wc', 'sudem_1000m_albers_curvature_wc',
                                            'composite_temp_min_1000m_k30_wc', 'composite_temp_max_1000m_k30_wc',
                                            'composite_rainfall_sum_1000m_k30_wc', 'composite_gdd_grapes_1000m_wc', 'composite_AETI_1000m_albers_wc',
                                            'nitrogen_0_to_5cm_1000m_albers_wc', 'sand_0_to_5cm_1000m_albers_wc', 'soc_0_to_5cm_1000m_albers_wc',
                                            'clay_content_0_to_5cm_1000m_albers_wc', 'soil_ph_0_to_5cm_1000m_albers_wc', 'silt_0_to_5cm_1000m_albers_wc',
                                            'ndvi_wc', 'euclidean_distance_roads_1000m_albers_wc', 'euclidean_distance_rivers_1000m_albers_wc']

def interpolate_columns(df, columns):
    for col in columns:
        df[col].interpolate(method='linear', inplace=True)
    return df

interpolate_columns(new_locations_df, columns_to_interpolate_for_new_locations)

In [ ]:
new_locations_df.isnull().sum()

In [ ]:
original_point_x = new_locations_df['POINT_X']
original_point_y = new_locations_df['POINT_Y']

In [ ]:
# Make predictions
columns_to_drop_2 = ['OID_', 'POINT_X', 'POINT_Y']

new_locations_df.drop(columns=columns_to_drop_2, inplace=True)
suitability_scores = xgb_model.predict(new_locations_df)

In [ ]:
# Add predictions to the new_locations dataframe
new_locations_df['suitability_score'] = suitability_scores

# Add POINT_X and POINT_Y columns back to the dataframe
new_locations_df['POINT_X'] = original_point_x
new_locations_df['POINT_Y'] = original_point_y

In [ ]:
# Save results
# If i can save it with point_x and point_y that would be great
new_locations_df.to_csv('wc_predicted_suitability.csv', index=False)

In [ ]:
wc_predicted_suitability = pd.read_csv('C:/Users/charl/Desktop/__Thesis - Research/Github/vineyard-establishment-using-xgboost/wc_predicted_suitability.csv')        

In [ ]:
wc_predicted_suitability['suitability_score'].describe()

In [ ]:
wc_vineyard_locations = wc_predicted_suitability['suitability_score']

In [ ]:
wc_vineyard_locations.to_csv('only_suitability_score_wc.csv', index=False)
new_locations_df_2

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
from scipy.interpolate import griddata
import rasterio
from rasterio.transform import from_origin

In [ ]:
location_columns = ['POINT_X', 'POINT_Y']
merged_df = pd.concat([wc_vineyard_locations, new_locations_df_2[location_columns]], axis=1)

merged_df

In [ ]:
merged_df.isnull().sum()

In [ ]:
merged_df

In [ ]:
# Step 4: Create a geometry column
# geometry = [Point(xy) for xy in zip(merged_df['POINT_X'], merged_df['POINT_Y'])]\
geometry =gpd.points_from_xy(merged_df['POINT_Y'], merged_df['POINT_X'])

In [ ]:
# Step 5: Create a GeoDataFrame
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry)

In [ ]:
# Step 6: Set the coordinate reference system (CRS)
gdf.set_crs(epsg=4326, inplace=True)

In [ ]:
# gdf.to_file('output_shapefile.shp')
print(gdf.crs)
print(gdf.total_bounds)
print(gdf.geometry.x.describe())
print(gdf.geometry.y.describe())

In [ ]:
# Define the grid
x_min, y_min, x_max, y_max = gdf.total_bounds

# Calculate the range of x and y
x_range = x_max - x_min
y_range = y_max - y_min

# Decide on a reasonable number of cells (e.g., 1000 x 1000)
n_cells = 1000*1000

# Calculate resolution
resolution_x = (x_max - x_min) / n_cells
resolution_y = (y_max - y_min) / n_cells

# Define tile size
tile_size = 100

print(f"Calculated resolution: X {resolution_x:.2f} Y {resolution_y:.2f}")


In [ ]:

 # Adjust this based on your desired output resolution
x = np.linspace(x_min, x_max, num=n_cells)
y = np.linspace(y_min, y_max, num=n_cells)
xx, yy = np.meshgrid(x, y)

In [ ]:
# Interpolate the suitability scores
points = gdf[['longitude', 'latitude']].values
values = gdf['suitability_score'].values
grid_z = griddata(points, values, (xx, yy), method='linear')

In [ ]:
# Define chunk size
chunk_size = 100

# Initialize empty grid
grid_z = np.zeros((n_cells, n_cells))

# Process in chunks
for i in range(0, n_cells, chunk_size):
    for j in range(0, n_cells, chunk_size):
        # Get chunk of grid
        xx_chunk = xx[i:i+chunk_size, j:j+chunk_size]
        yy_chunk = yy[i:i+chunk_size, j:j+chunk_size]
        
        # Interpolate for this chunk
        grid_z_chunk = griddata(points, values, (xx_chunk, yy_chunk), method='linear')
        
        # Assign to main grid
        grid_z[i:i+chunk_size, j:j+chunk_size] = grid_z_chunk

print("Interpolation complete")

In [ ]:
# Save as a raster
transform = from_origin(x_min, y_max, resolution, resolution)
new_dataset = rasterio.open('suitability_map.tif', 'w', driver='GTiff',
                            height=grid_z.shape[0], width=grid_z.shape[1],
                            count=1, dtype=grid_z.dtype,
                            crs=gdf.crs, transform=transform)
new_dataset.write(grid_z, 1)
new_dataset.close()